# Лабораторная работа №2 — логистическая и линейная регрессии

## 1. Выбор начальных условий

- **Spambase:** бинарная классификация писем по 57 признакам. F1-macro — основная метрика с равным весом классов; accuracy — доля верных ответов. Используем то же стратифицированное разбиение 80/20.
- **SML2010:** прогноз температуры в столовой на 15 минут вперёд. MAE показывает среднюю ошибку в °C, RMSE сильнее отзывается на крупные ошибки. Используем то же хронологическое разбиение.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from data_setup import DINING_TEMPERATURE, load_sml2010, load_spambase, sml2010_cv, spambase_cv

Xc_train, Xc_test, yc_train, yc_test = load_spambase()
Xr_train, Xr_test, yr_train, yr_test = load_sml2010()
print(f"Spambase: {Xc_train.shape} / {Xc_test.shape}")
print(f"SML2010:  {Xr_train.shape} / {Xr_test.shape}")

Spambase: (3363, 57) / (841, 57)
SML2010:  (2763, 18) / (1370, 18)


## 2. Создание бейзлайна и оценка качества

Для классификации используем логистическую регрессию с L2-регуляризацией (`C=1`). Стандартизация входит уже в бейзлайн: она выравнивает масштабы признаков для оптимизации и применения L2. Для регрессии используем обычный метод наименьших квадратов на всех 18 признаках. 

In [2]:
def classification_metrics(y, prediction):
    return {"f1_macro": f1_score(y, prediction, average="macro"), "accuracy": accuracy_score(y, prediction)}


def regression_metrics(y, prediction):
    return {"mae": mean_absolute_error(y, prediction), "rmse": root_mean_squared_error(y, prediction)}


clf_pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
clf_base = clf_pipeline.fit(Xc_train, yc_train)
reg_base = LinearRegression().fit(Xr_train, yr_train)
classification_scores = {"sklearn_base": classification_metrics(yc_test, clf_base.predict(Xc_test))}
regression_scores = {"sklearn_base": regression_metrics(yr_test, reg_base.predict(Xr_test))}
persistence = regression_metrics(yr_test, Xr_test[DINING_TEMPERATURE])

display(pd.DataFrame.from_dict(classification_scores, orient="index"))
display(pd.DataFrame.from_dict({**regression_scores, "текущая температура": persistence}, orient="index"))

,f1_macro,accuracy
sklearn_base,0.920618,0.9239


,mae,rmse
sklearn_base,0.020732,0.028411
текущая температура,0.116360,0.133181


## 3. Улучшение бейзлайна

### Гипотезы

- **Классификация:** изменение силы L2-регуляризации может улучшить обобщение. Меньшее `C` сильнее штрафует большие коэффициенты. Проверяем `C ∈ {0.01, 0.1, 1, 10}`.
- **Регрессия:** на горизонте 15 минут трёх температур (столовая, комната, улица) может хватить; отбор снизит влияние косвенных признаков, таких как освещённость. Квадраты и попарные произведения этих температур могут описать нелинейные зависимости. Сравниваем полную модель, модель на трёх температурах и модель на них же с признаками второй степени.

### Проверка гипотез

Оцениваем версии только на обучающей части: F1-macro по трём стратифицированным фолдам и MAE по трём временным фолдам с зазором. Преобразования остаются внутри `Pipeline`. Отрицательную метрику `neg_mean_absolute_error` переводим обратно в MAE, где меньше — лучше.

In [3]:
clf_search = GridSearchCV(
    clf_pipeline, {"logisticregression__C": [0.01, 0.1, 1, 10]},
    cv=spambase_cv(), scoring="f1_macro",
).fit(Xc_train, yc_train)
display(pd.DataFrame({
    "C": clf_search.cv_results_["param_logisticregression__C"],
    "CV F1-macro": clf_search.cv_results_["mean_test_score"],
}).sort_values("CV F1-macro", ascending=False))

temperatures = [DINING_TEMPERATURE, "Temperature_Habitacion_Sensor", "Temperature_Exterior_Sensor"]
reg_configs = {
    "все признаки": (Xr_train.columns, 1),
    "три температуры": (temperatures, 1),
    "три температуры + квадратичные признаки": (temperatures, 2),
}


def linear_pipeline(degree):
    features = PolynomialFeatures(degree=2, include_bias=False) if degree == 2 else "passthrough"
    return Pipeline([("features", features), ("model", LinearRegression())])


reg_cv_mae = {}
for name, (columns, degree) in reg_configs.items():
    scores = cross_val_score(
        linear_pipeline(degree), Xr_train[columns], yr_train,
        cv=sml2010_cv(), scoring="neg_mean_absolute_error",
    )
    reg_cv_mae[name] = -scores.mean()
display(pd.Series(reg_cv_mae, name="CV MAE, °C").sort_values().to_frame())

,C,CV F1-macro
3,10.00,0.921895
2,1.00,0.920308
1,0.10,0.916586
0,0.01,0.900315


,"CV MAE, °C"
все признаки,0.028886
три температуры,0.069472
три температуры + квадратичные признаки,0.069557


### 3c. Улучшенный бейзлайн

Берём `C` с лучшим F1-macro и регрессионную конфигурацию с минимальным CV MAE. Модели обучены на всей обучающей части; теперь оцениваем их на неизменном тесте и сравниваем с пунктом 2.

In [4]:
best_c = clf_search.best_params_["logisticregression__C"]
best_reg = min(reg_cv_mae, key=reg_cv_mae.get)
best_columns, best_degree = reg_configs[best_reg]
reg_improved = linear_pipeline(best_degree).fit(Xr_train[best_columns], yr_train)

print(f"Spambase: C={best_c}")
print(f"SML2010: {best_reg}")
classification_scores["sklearn_improved"] = classification_metrics(yc_test, clf_search.predict(Xc_test))
regression_scores["sklearn_improved"] = regression_metrics(yr_test, reg_improved.predict(Xr_test[best_columns]))
display(pd.DataFrame.from_dict(classification_scores, orient="index"))
display(pd.DataFrame.from_dict(regression_scores, orient="index"))

Spambase: C=10
SML2010: все признаки


,f1_macro,accuracy
sklearn_base,0.920618,0.923900
sklearn_improved,0.925580,0.928656


,mae,rmse
sklearn_base,0.020732,0.028411
sklearn_improved,0.020732,0.028411


**Выводы.** Для Spambase лучшим по CV оказалось `C=10`: на тесте F1-macro вырос с 0,9206 до 0,9256. Для SML2010 отбор трёх температур ухудшил CV MAE с 0,0289 до 0,0695 °C; квадратичные признаки не вернули потерянное качество (0,0696 °C). Оставляем все признаки без преобразований: тестовый MAE 0,0207 °C, лучше прогноза по текущей температуре (0,1164 °C) на этом периоде.

## 4. Имплементация алгоритма машинного обучения

### Базовая реализация

**Логистическая регрессия.** Сигмоида переводит линейную комбинацию признаков в вероятность спама. Обучаем коэффициенты и свободный член методом градиентного спуска по средней логистической потере. Для соответствия шкале `C` в sklearn добавляем к градиенту коэффициентов `w / (C × n)`; свободный член не штрафуем. Используем фиксированный шаг и число итераций; это проще, чем оптимизатор sklearn, но не гарантирует ту же точку остановки. Предсказываем класс `1`, когда вероятность выше 0,5.

**Линейная регрессия.** Добавляем столбец единиц для свободного члена и решаем задачу наименьших квадратов через `np.linalg.lstsq` — без обращения матрицы и без готовой модели sklearn.

In [5]:
class LogisticRegressionNumPy:
    def __init__(self, C=1.0, learning_rate=0.1, steps=15000):
        self.C = C
        self.learning_rate = learning_rate
        self.steps = steps

    @staticmethod
    def sigmoid(logits):
        return 1 / (1 + np.exp(-np.clip(logits, -50, 50)))

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y)
        n, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        for _ in range(self.steps):
            error = self.sigmoid(X @ self.weights + self.bias) - y
            grad = X.T @ error / n + self.weights / (self.C * n)
            self.weights -= self.learning_rate * grad
            self.bias -= self.learning_rate * error.mean()
        return self

    def predict(self, X):
        probability = self.sigmoid(np.asarray(X, dtype=float) @ self.weights + self.bias)
        return (probability > 0.5).astype(int)


class LinearRegressionNumPy:
    @staticmethod
    def with_bias(X):
        return np.column_stack((np.ones(len(X)), np.asarray(X, dtype=float)))

    def fit(self, X, y):
        self.weights = np.linalg.lstsq(self.with_bias(X), np.asarray(y), rcond=None)[0]
        return self

    def predict(self, X):
        return self.with_bias(X) @ self.weights

Проверяем классы на простых разделимых точках и прогноз на известной прямой. Затем воспроизводим обработку бейзлайна: стандартизируем признаки писем, а для температуры используем все исходные признаки. `StandardScaler` обучаем только на тренировочных данных.

In [6]:
toy_clf = LogisticRegressionNumPy().fit([[-2], [-1], [1], [2]], [0, 0, 1, 1])
np.testing.assert_array_equal(toy_clf.predict([[-3], [3]]), [0, 1])
toy_reg = LinearRegressionNumPy().fit([[0], [1], [2]], [1, 3, 5])
np.testing.assert_allclose(toy_reg.predict([[3], [4]]), [7, 9])

scaler = StandardScaler().fit(Xc_train)
Xc_train_scaled, Xc_test_scaled = scaler.transform(Xc_train), scaler.transform(Xc_test)
clf_own = LogisticRegressionNumPy().fit(Xc_train_scaled, yc_train)
reg_own = LinearRegressionNumPy().fit(Xr_train, yr_train)
classification_scores["custom_base"] = classification_metrics(yc_test, clf_own.predict(Xc_test_scaled))
regression_scores["custom_base"] = regression_metrics(yr_test, reg_own.predict(Xr_test))
base_variants = ["sklearn_base", "custom_base"]
display(pd.DataFrame.from_dict(classification_scores, orient="index").loc[base_variants])
display(pd.DataFrame.from_dict(regression_scores, orient="index").loc[base_variants])

,f1_macro,accuracy
sklearn_base,0.920618,0.923900
custom_base,0.921819,0.925089


,mae,rmse
sklearn_base,0.020732,0.028411
custom_base,0.020732,0.028411


**Сравнение с пунктом 2.** Базовая логистическая регрессия NumPy дала близкие метрики: F1-macro 0,9218 против 0,9206. Базовые линейные модели совпали по MAE и RMSE с точностью вывода.

### Улучшенная реализация

Для логистической регрессии переносим лучшее `C` из пункта 3. Для линейной берём те же столбцы и, если выбрана степень 2, строим квадратичные признаки. Сравниваем обе собственные модели с улучшенными вариантами sklearn.

In [7]:
clf_own_improved = LogisticRegressionNumPy(C=best_c).fit(Xc_train_scaled, yc_train)
Xr_train_best, Xr_test_best = Xr_train[best_columns], Xr_test[best_columns]
if best_degree == 2:
    poly = PolynomialFeatures(degree=2, include_bias=False).fit(Xr_train_best)
    Xr_train_best, Xr_test_best = poly.transform(Xr_train_best), poly.transform(Xr_test_best)
reg_own_improved = LinearRegressionNumPy().fit(Xr_train_best, yr_train)

classification_scores["custom_improved"] = classification_metrics(yc_test, clf_own_improved.predict(Xc_test_scaled))
regression_scores["custom_improved"] = regression_metrics(yr_test, reg_own_improved.predict(Xr_test_best))
improved_variants = ["sklearn_improved", "custom_improved"]
display(pd.DataFrame.from_dict(classification_scores, orient="index").loc[improved_variants])
display(pd.DataFrame.from_dict(regression_scores, orient="index").loc[improved_variants])

,f1_macro,accuracy
sklearn_improved,0.925580,0.928656
custom_improved,0.919337,0.922711


,mae,rmse
sklearn_improved,0.020732,0.028411
custom_improved,0.020732,0.028411


**Сравнение с пунктом 3.** Для `C=10` собственная логистическая регрессия получила F1-macro 0,9193 против 0,9256 у sklearn. Перенос `C` не улучшил её тестовый результат относительно собственной базовой версии. Собственный градиентный спуск останавливается после фиксированного числа шагов, а sklearn использует другой оптимизатор; совпадения коэффициентов не ожидаем. Для линейной регрессии лучшей осталась базовая конфигурация: обе реализации дали MAE 0,0207 °C.

In [8]:
variants = ["sklearn_base", "sklearn_improved", "custom_base", "custom_improved"]
rows = []
for task, algorithm, scores in [
    ("classification", "LogisticRegression", classification_scores),
    ("regression", "LinearRegression", regression_scores),
]:
    for variant in variants:
        rows.append({"lab": 2, "task": task, "algorithm": algorithm, "variant": variant, **scores[variant]})

results = pd.DataFrame(rows)
results.to_csv(root / "results" / "lab02.csv", index=False, float_format="%.8f")
for task in results["task"].unique():
    display(results.loc[results["task"] == task].dropna(axis=1, how="all").reset_index(drop=True))

,lab,task,algorithm,variant,f1_macro,accuracy
0,2,classification,LogisticRegression,sklearn_base,0.920618,0.923900
1,2,classification,LogisticRegression,sklearn_improved,0.925580,0.928656
2,2,classification,LogisticRegression,custom_base,0.921819,0.925089
3,2,classification,LogisticRegression,custom_improved,0.919337,0.922711


,lab,task,algorithm,variant,mae,rmse
0,2,regression,LinearRegression,sklearn_base,0.020732,0.028411
1,2,regression,LinearRegression,sklearn_improved,0.020732,0.028411
2,2,regression,LinearRegression,custom_base,0.020732,0.028411
3,2,regression,LinearRegression,custom_improved,0.020732,0.028411
